## Quantizing using PEFT QLoRa

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
# Configuration
model_id = "tiiuae/falcon-7b"
dataset_name = "viggo"

In [ ]:
# QLoRA 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
#Load Model and Tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Prepare Model for Training
model = prepare_model_for_kbit_training(model)

In [ ]:
# Lora Config
peft_config = LoraConfig(
    r=16, # Rank
    lora_alpha=32,
    target_modules=["query_key_value"], # Specific to Falcon- chnage value
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [ ]:
# Load Dataset
data = load_dataset(dataset_name)
def tokenize(sample):
    return tokenizer(sample["text"], padding="max_length", truncation=True, max_length=512)

tokenized_data = data.map(tokenize, batched=True)

In [ ]:
#Training Arguments
training_args = TrainingArguments(
    output_dir="./qlora_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    fp16=True,
    optim="paged_adamw_8bit" # optimizer for QLoRA
)

In [ ]:
# 8. Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_data["train"],
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)


In [ ]:
#Train
model.config.use_cache = False
trainer.train()
